# DeiT Attention Rollout (heatmap)
Heatmap dựa trên attention rollout của ViT để highlight vùng ảnh quan trọng.

In [ ]:
cd ..

In [ ]:
import sys
import json

import torch
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import OmegaConf

from src.baseline.data import MRIVolumeJPGDataset
from src.baseline.models.deit2d_classifier import DeiT2DClassifier

In [ ]:
config_path = 'configs/baseline/train_2d_oasis.yaml'
cfg = OmegaConf.load(config_path)
print(f"Loaded config from {config_path}")

fold_index = 2
split_path = cfg.data.split_file
with open(split_path, 'r') as f:
    split_data = json.load(f)
print(f"Loaded split data from {split_path}")
fold = split_data['folds'][fold_index]
val_items = fold['val']
val_subject_ids = [
    (item['subject_id'] if isinstance(item, dict) else item)
    for item in val_items
]

ckpt_dir = f"checkpoints/deit_base_attn_120sl_oasis_5folds/fold_{fold_index}"
ckpt_path = f"{ckpt_dir}/last.ckpt"

output_dir = f"outputs/attn_rollout"

print('Checkpoint:', ckpt_path)
print('Output dir:', output_dir)
print('Val subjects:', len(val_subject_ids))


In [ ]:
val_ds = MRIVolumeJPGDataset(
    root_dir=cfg.data.train_dir,
    classes=list(cfg.data.classes),
    view=getattr(cfg.data, 'jpg_view', 'ax'),
    num_slices=int(cfg.data.num_slices),
    slice_strategy=getattr(cfg.data, 'val_strategy', 'uniform'),
    image_size=int(cfg.data.image_size),
    augment=False,
    normalize=True,
    subject_ids=set(val_subject_ids),
)

val_loader = torch.utils.data.DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print('Val subjects:', len(val_ds))


In [ ]:
# Load model directly (no Lightning)
model = DeiT2DClassifier(
    timm_name=cfg.model.timm_name,
    pretrained=False,
    image_size=cfg.data.image_size,
    in_channels=cfg.model.in_channels,
    num_classes=cfg.model.num_classes,
    drop_path_rate=cfg.model.drop_path_rate,
    attn_drop_rate=cfg.model.attn_drop_rate,
    dropout=cfg.model.dropout,
)
ckpt = torch.load(ckpt_path, map_location='cpu')
state = ckpt.get('state_dict', ckpt)
state = {k.replace('model.', ''): v for k, v in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
print('Missing:', missing)
print('Unexpected:', unexpected)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()
print('Device:', device)


In [ ]:
# Attention rollout (compute attn from qkv)
def compute_rollout(attn_mats):
    result = None
    for attn in attn_mats:
        attn = attn.mean(dim=1)  # avg heads
        attn = attn / attn.sum(dim=-1, keepdim=True)
        if result is None:
            result = attn
        else:
            result = attn @ result
    return result

def attention_rollout(model, x):
    attn_mats = []
    hooks = []

    def make_hook(blk):
        def hook_fn(module, inp, out):
            # out: [B, N, 3*D]
            qkv = out
            B, N, threeD = qkv.shape
            num_heads = blk.attn.num_heads
            dim = threeD // 3
            head_dim = dim // num_heads
            qkv = qkv.reshape(B, N, 3, num_heads, head_dim).permute(2, 0, 3, 1, 4)
            q, k, v = qkv[0], qkv[1], qkv[2]
            attn = (q @ k.transpose(-2, -1)) * blk.attn.scale
            attn = attn.softmax(dim=-1)
            attn_mats.append(attn.detach())
        return hook_fn

    for blk in model.encoder.blocks:
        hooks.append(blk.attn.qkv.register_forward_hook(make_hook(blk)))

    with torch.no_grad():
        _ = model.encoder(x)

    for h in hooks:
        h.remove()

    if not attn_mats:
        raise RuntimeError('Không lấy được attention mats từ qkv.')

    rollout = compute_rollout(attn_mats)
    return rollout


In [ ]:
# Attn rollout cho TẤT CẢ slices, lưu GIF + cache rollouts/weights
import os
import matplotlib.cm as cm
import imageio.v2 as imageio

os.makedirs(output_dir, exist_ok=True)

alz_index = cfg.data.classes.index('alzheimer') if 'alzheimer' in cfg.data.classes else 1
alz_batch = None
for xb, yb in val_loader:
    if int(yb.item()) == alz_index:
        alz_batch = (xb, yb)
        break
if alz_batch is None:
    raise RuntimeError('Không tìm thấy subject lớp Alzheimer trong val_loader')
x, y = alz_batch
x = x.to(device)  # (B, S, C, H, W)

with torch.no_grad():
    b, s, c, h, w = x.shape
    feats = model.encoder(x.view(b * s, c, h, w))
    feats = feats.view(b, s, -1)
    weights = model.pool.attn(feats)  # (B, S, 1)
    weights = torch.softmax(weights, dim=1).squeeze(-1)

slice_weights = weights[0].detach().cpu().numpy()
slice_overlays = []
frames = []
for i in range(s):
    img_tensor = x[:, i]  # (B, C, H, W)
    rollout = attention_rollout(model, img_tensor)
    mask = rollout[0, 0, 1:]
    h_m = w_m = int(mask.numel() ** 0.5)
    mask = mask.reshape(h_m, w_m).cpu().numpy()
    mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
    mask = np.kron(mask, np.ones((16, 16)))
    mask = mask[: h, : w]

    img_raw = img_tensor[0].detach().cpu().permute(1, 2, 0).numpy()
    img_norm = (img_raw - img_raw.min()) / (img_raw.max() - img_raw.min() + 1e-8)
    if img_norm.shape[-1] == 1:
        img_norm = img_norm[:, :, 0]

    heat = cm.jet(mask)[..., :3]  # RGB
    img_rgb = np.stack([img_norm]*3, axis=-1)
    overlay = (0.6 * img_rgb + 0.4 * heat).clip(0, 1)
    slice_overlays.append(overlay)
    frames.append((overlay * 255).astype(np.uint8))

out_path = f"{output_dir}/alz_attention_rollout_fold{fold_index}.gif"
imageio.mimsave(out_path, frames, duration=0.08)
print('Saved:', out_path)


In [ ]:
# Top-weighted slices (attention pooling weights) + attn rollout overlay + save grid
import os
import math
import matplotlib.cm as cm

os.makedirs(output_dir, exist_ok=True)

# Fallback compute weights if not cached from GIF cell
if 'slice_weights' not in globals():
    with torch.no_grad():
        b, s, c, h, w = x.shape
        feats = model.encoder(x.view(b * s, c, h, w))
        feats = feats.view(b, s, -1)
        weights = model.pool.attn(feats)  # (B, S, 1)
        weights = torch.softmax(weights, dim=1).squeeze(-1)
    slice_weights = weights[0].detach().cpu().numpy()

weights_np = slice_weights
top_k = min(8, x.shape[1])
top_idx = np.argsort(weights_np)[::-1][:top_k]

def _overlay_for_idx(idx: int):
    if 'slice_overlays' in globals() and len(slice_overlays) == x.shape[1]:
        return slice_overlays[idx]
    img_tensor = x[:, idx]  # (B, C, H, W)
    rollout = attention_rollout(model, img_tensor)
    mask = rollout[0, 0, 1:]
    h_m = w_m = int(mask.numel() ** 0.5)
    mask = mask.reshape(h_m, w_m).cpu().numpy()
    mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
    mask = np.kron(mask, np.ones((16, 16)))
    mask = mask[: x.shape[3], : x.shape[4]]

    img_raw = img_tensor[0].detach().cpu().permute(1, 2, 0).numpy()
    img_norm = (img_raw - img_raw.min()) / (img_raw.max() - img_raw.min() + 1e-8)
    if img_norm.shape[-1] == 1:
        img_norm = img_norm[:, :, 0]

    heat = cm.jet(mask)[..., :3]
    img_rgb = np.stack([img_norm] * 3, axis=-1)
    return (0.6 * img_rgb + 0.4 * heat).clip(0, 1)

cols = 4
rows = math.ceil(top_k / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
axes = np.array(axes).reshape(-1)

for ax_i, ax in enumerate(axes):
    ax.axis('off')
    if ax_i >= top_k:
        continue
    idx = int(top_idx[ax_i])
    overlay = _overlay_for_idx(idx)
    ax.imshow(overlay)
    ax.set_title(f"slice {idx} | w={weights_np[idx]:.3f}", fontsize=9)

grid_path = f"{output_dir}/top_weighted_slices_rollout_fold{fold_index}.png"
plt.tight_layout()
plt.savefig(grid_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', grid_path)


In [ ]:
# Slice weight curve (temporal focus)
import os
import numpy as np
import matplotlib.pyplot as plt

os.makedirs(output_dir, exist_ok=True)

# Fallback compute weights if not cached
if 'slice_weights' not in globals():
    with torch.no_grad():
        b, s, c, h, w = x.shape
        feats = model.encoder(x.view(b * s, c, h, w))
        feats = feats.view(b, s, -1)
        weights = model.pool.attn(feats)  # (B, S, 1)
        weights = torch.softmax(weights, dim=1).squeeze(-1)
    slice_weights = weights[0].detach().cpu().numpy()

w = slice_weights.astype(float)
w = w / (w.sum() + 1e-12)
x_idx = np.arange(len(w))

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(x_idx, w, linewidth=1.6)

top_k = min(8, len(w))
top_idx = np.argsort(w)[::-1][:top_k]
ax.scatter(top_idx, w[top_idx], color='red', s=25, label='top')

ax.set_xlabel('Slice index')
ax.set_ylabel('Weight (softmax)')
ax.set_title('Slice weight curve')
ax.legend()

curve_path = f"{output_dir}/slice_weight_curve_fold{fold_index}.png"
plt.tight_layout()
plt.savefig(curve_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', curve_path)
